### Reading the pdf file

In [1]:
from langchain_community.document_loaders import PyPDFLoader

# 1. Initialize the loader with the file path
loader = PyPDFLoader("docs/oracle_project_costing.pdf")

# 2. Load the document pages into memory
pages = loader.load()

# 3. Access the data
for page in pages:
    print(f"--- Page {page.metadata['page']} ---")
    print(page.page_content[:200]) 

/tmp/ipykernel_6772/101711017.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


--- Page 0 ---
Oracle® Project Costing
User Guide
Release 12.2
 Part No. E48918-19
November 2022
--- Page 1 ---
Oracle Project Costing User Guide, Release 12.2
Part No. E48918-19
Copyright © 1994, 2022, Oracle and/or its affiliates. 
Primary Author:     Sudha Seshadri, Pooja Vijay Kumar, Stacey Tucker-Blosch, K
--- Page 2 ---
iii
 
Contents
Send Us Your Comments
Preface
1 Overview of Project Costing
--- Page 3 ---
iv
2 Oracle Projects Command Center
3 Budgets
--- Page 4 ---
v
Accounting For Burden and Total Burdened Cost Encumbrances in Oracle
Creating Project Budgets for Top-Down Budget Integration with Oracle Contract
4 Expenditures
--- Page 5 ---
vi
--- Page 6 ---
vii
5 Burdening
--- Page 7 ---
viii
6 Allocations
7 Asset Capitalization
--- Page 8 ---
ix
--- Page 9 ---
x
8 Cross Charge
9 Integration with Other Oracle Applications
Integrating with Oracle Purchasing and Oracle Payables (Requisitions, Purchase Orders, and
--- Page 10 ---
xi
--- Page 11 ---
xii
Index
--- Page 12 ---
xiii

### Chunking - Semantic Chunking

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings
import os
from dotenv import load_dotenv
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


/tmp/ipykernel_6772/3817903839.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


In [ ]:
# Initialize your embedding model
# embeddings = OpenAIEmbeddings(
#     api_key=OPENAI_API_KEY,
#     model="text-embedding-3-large"
# )

In [4]:
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
embed_model = FastEmbedEmbeddings(model_name="BAAI/bge-base-en-v1.5")

semantic_chunks = []
semantic_chunker = SemanticChunker(embed_model, breakpoint_threshold_type="percentile")

for page in pages:
    chunks = semantic_chunker.create_documents([page.page_content])

    for chunk_id, chunk in enumerate(chunks):
        chunk.metadata = page.metadata.copy()
        chunk.metadata["chunk_id"] = chunk_id
        semantic_chunks.append(chunk)

print(f"Total Semantic Chunks: {len(semantic_chunks)}")

/run/media/abhisek-dattta/My Space/chargeability-analytics/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files: 100%|██████████| 5/5 [01:17<00:00, 15.45s/it]


Total Semantic Chunks: 1315


In [6]:
pages

[Document(metadata={'producer': 'Oracle BI Publisher 12.2.1.2.0', 'creator': 'PyPDF', 'creationdate': '', 'type': '/Info', 'source': 'docs/oracle_project_costing.pdf', 'total_pages': 668, 'page': 0, 'page_label': '1'}, page_content='Oracle® Project Costing\nUser Guide\nRelease 12.2\n Part No. E48918-19\nNovember 2022'),
 Document(metadata={'producer': 'Oracle BI Publisher 12.2.1.2.0', 'creator': 'PyPDF', 'creationdate': '', 'type': '/Info', 'source': 'docs/oracle_project_costing.pdf', 'total_pages': 668, 'page': 1, 'page_label': '2'}, page_content='Oracle Project Costing User Guide, Release 12.2\nPart No. E48918-19\nCopyright © 1994, 2022, Oracle and/or its affiliates. \nPrimary Author:     Sudha Seshadri, Pooja Vijay Kumar, Stacey Tucker-Blosch, Kevin Brown\nContributing Author:     Shyam Guduru, Anupam Johri, Sujan Korrapati, Prem Subramanian, Seema Singh, \nUmamaheswari Subramanian\nThis software and related documentation are provided under a license agreement containing restriction

In [5]:
semantic_chunks

[Document(metadata={'producer': 'Oracle BI Publisher 12.2.1.2.0', 'creator': 'PyPDF', 'creationdate': '', 'type': '/Info', 'source': 'docs/oracle_project_costing.pdf', 'total_pages': 668, 'page': 0, 'page_label': '1', 'chunk_id': 0}, page_content='Oracle® Project Costing\nUser Guide\nRelease 12.2\n Part No. E48918-19\nNovember 2022'),
 Document(metadata={'producer': 'Oracle BI Publisher 12.2.1.2.0', 'creator': 'PyPDF', 'creationdate': '', 'type': '/Info', 'source': 'docs/oracle_project_costing.pdf', 'total_pages': 668, 'page': 1, 'page_label': '2', 'chunk_id': 0}, page_content='Oracle Project Costing User Guide, Release 12.2\nPart No. E48918-19\nCopyright © 1994, 2022, Oracle and/or its affiliates. Primary Author:     Sudha Seshadri, Pooja Vijay Kumar, Stacey Tucker-Blosch, Kevin Brown\nContributing Author:     Shyam Guduru, Anupam Johri, Sujan Korrapati, Prem Subramanian, Seema Singh, \nUmamaheswari Subramanian\nThis software and related documentation are provided under a license agre

### Chuck to VectorDB - Qdrant

In [11]:
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
QDRANT_ENDPOINT = os.getenv("QDRANT_ENDPOINT")

from qdrant_client import QdrantClient

qdrant_client = QdrantClient(
    url= QDRANT_ENDPOINT,
    api_key= QDRANT_API_KEY
)

print(qdrant_client.get_collections())

collections=[]


### Creating collection

In [20]:
from qdrant_client.models import VectorParams, Distance


COLLECTION_NAME = "pdf_documents"


if COLLECTION_NAME not in [
    c.name for c in qdrant_client.get_collections().collections
]:

    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=768,
            distance=Distance.COSINE,
            timeout=300
        )
    )

### Creating VDB

In [21]:
from langchain_qdrant import QdrantVectorStore


vector_store = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION_NAME,
    embedding=embed_model,
    
)

### Ingesting chunks

chunk.page_content
        |
        |
embed_model.embed_documents()
        |
        |
768 dimension vector
        |
        |
Qdrant insert

In [22]:
import time

batch_size = 50

for i in range(0, len(semantic_chunks), batch_size):

    batch = semantic_chunks[i:i+batch_size]

    success = False

    while not success:
        try:
            vector_store.add_documents(batch)
            success = True

            print(
                f"Uploaded {min(i+batch_size, len(semantic_chunks))}/{len(semantic_chunks)}"
            )

        except Exception as e:
            print("Upload failed:", e)
            print("Retrying in 10 seconds...")
            time.sleep(10)

Uploaded 50/1315
Uploaded 100/1315
Uploaded 150/1315
Uploaded 200/1315
Uploaded 250/1315
Uploaded 300/1315
Uploaded 350/1315
Uploaded 400/1315
Uploaded 450/1315
Uploaded 500/1315
Uploaded 550/1315
Uploaded 600/1315
Uploaded 650/1315
Uploaded 700/1315
Uploaded 750/1315
Uploaded 800/1315
Upload failed: The write operation timed out
Retrying in 10 seconds...
Upload failed: The write operation timed out
Retrying in 10 seconds...
Upload failed: The write operation timed out
Retrying in 10 seconds...
Uploaded 850/1315
Uploaded 900/1315
Uploaded 950/1315
Uploaded 1000/1315
Uploaded 1050/1315
Uploaded 1100/1315
Uploaded 1150/1315
Uploaded 1200/1315
Uploaded 1250/1315
Uploaded 1300/1315
Uploaded 1315/1315


### For deleting

In [ ]:
# qdrant_client.delete_collection(
#     collection_name="pdf_documents"
# )

# print("Collection deleted")

Collection deleted


### Retrieval (after restart the kernel)

In [1]:
from dotenv import load_dotenv
import os

from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings

/run/media/abhisek-dattta/My Space/chargeability-analytics/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_21779/1880382027.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings.fastembed import FastEmbedEmbeddings


In [2]:
load_dotenv()

QDRANT_ENDPOINT = os.getenv("QDRANT_ENDPOINT")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

### Connect to Qdrant

In [3]:
qdrant_client = QdrantClient(
    url=QDRANT_ENDPOINT,
    api_key=QDRANT_API_KEY,
)

### Connect to the exsisting collection

In [4]:
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
embed_model = FastEmbedEmbeddings(model_name="BAAI/bge-base-en-v1.5")

Fetching 5 files: 100%|██████████| 5/5 [01:29<00:00, 17.86s/it]


In [5]:
vector_store = QdrantVectorStore(
    client=qdrant_client,
    collection_name="pdf_documents",
    embedding=embed_model,
)

### Create the retriever

In [9]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

### Querying

In [10]:
query = "can you tell me what is the chargeability analytics project about?"

docs = retriever.invoke(query)

for doc in docs:
    print(doc.metadata)
    print(doc.page_content)
    print("=" * 80)

{'producer': 'Oracle BI Publisher 12.2.1.2.0', 'creator': 'PyPDF', 'creationdate': '', 'type': '/Info', 'source': 'docs/oracle_project_costing.pdf', 'total_pages': 668, 'page': 299, 'page_label': '300', 'chunk_id': 1, '_id': 'cb3bcb67-afac-4f1a-b28c-d8a4160f6b36', '_collection_name': 'pdf_documents'}
Specifying Effective Dates for Transaction Controls
You can define transactions as chargeable for a given date range by entering an 
Effective From and Effective To date for each transaction control record. You must 
specify a start date; Oracle Projects defaults this value to the Effective From date of the 
project or task. The Effective To date is optional. Determining if an Item is Chargeable
Oracle Projects checks all levels of chargeability control when you try to charge a 
transaction to a project. The check is performed when you save the record. Oracle 
Projects checks the control when you:
• enter an online or pre-approved expenditure item
• copy a pre-approved timecard item
• tran